# Oil Price Prediction from Strategic Strait Shipping Data

**Pipeline overview:**
1. Generate realistic synthetic AIS shipping data for 6 strategic straits (2015–2023)
2. Fetch real WTI crude oil prices from FRED
3. Merge on date → feature engineer → train/test split
4. Train XGBoost, Random Forest, and Gradient Boosting models
5. Predict on test set → save results to CSV

> **Note on real AIS data:** Marine Cadastre (marinecadastre.gov) covers US coastal waters only.
> For the international straits below you need a global AIS feed.
> Recommended sources: [AISHub](https://www.aishub.net) (free/contributor),
> [UN Global Platform AIS](https://unstats.un.org/wiki/display/AIS) (free/academic),
> [Spire Maritime](https://spire.com/maritime) or [MarineTraffic](https://www.marinetraffic.com) (paid).
> To swap in real data: load your CSVs, apply `filter_to_straits()` in Cell 3, aggregate to daily
> counts per strait, then continue from Cell 4 — nothing else changes.

| Strait | Lat range | Lon range |
|---|---|---|
| Strait of Hormuz | 25.5 – 27.0 N | 55.5 – 57.5 E |
| Bab el-Mandeb | 11.5 – 13.0 N | 42.5 – 44.0 E |
| Suez Canal | 29.5 – 31.5 N | 32.0 – 33.0 E |
| Panama Canal | 8.5 – 9.5 N | 80.0 – 79.5 W |
| Strait of Malacca | 1.0 – 6.0 N | 99.0 – 105.0 E |
| Turkish Straits | 40.0 – 41.5 N | 26.0 – 29.5 E |

## 0. Install Dependencies

In [ ]:
!pip install pandas numpy scikit-learn xgboost pandas-datareader matplotlib requests tqdm -q

## 1. Imports & Config

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from functools import reduce

warnings.filterwarnings('ignore')
np.random.seed(42)

STRAITS = {
    'Hormuz':          dict(lat_min=25.5, lat_max=27.0, lon_min=55.5,  lon_max=57.5),
    'Bab_el_Mandeb':   dict(lat_min=11.5, lat_max=13.0, lon_min=42.5,  lon_max=44.0),
    'Suez_Canal':      dict(lat_min=29.5, lat_max=31.5, lon_min=32.0,  lon_max=33.0),
    'Panama_Canal':    dict(lat_min= 8.5, lat_max= 9.5, lon_min=-80.0, lon_max=-79.5),
    'Malacca':         dict(lat_min= 1.0, lat_max= 6.0, lon_min=99.0,  lon_max=105.0),
    'Turkish_Straits': dict(lat_min=40.0, lat_max=41.5, lon_min=26.0,  lon_max=29.5),
}

DATE_START = '2015-01-01'
DATE_END   = '2023-12-31'
print('Config ready. Straits:', list(STRAITS.keys()))

## 2. Fetch Real WTI Crude Oil Prices from FRED

In [ ]:
import pandas_datareader as web

wti_raw = web.DataReader('DCOILWTICO', 'fred', '2009-01-01', '2025-12-31')
wti_raw = wti_raw.reset_index()
wti_raw.columns = ['date', 'oil_price_wti']
wti_raw['date'] = pd.to_datetime(wti_raw['date'])

# Forward-fill weekends / public holidays then drop remaining NaNs
wti = (
    wti_raw
    .set_index('date')
    .resample('D').ffill()
    .reset_index()
    .dropna(subset=['oil_price_wti'])
)

print(f'WTI rows : {len(wti):,}')
print(f'Range    : {wti["date"].min().date()} to {wti["date"].max().date()}')
wti.tail()

## 3. Build Synthetic AIS Shipping Data

Simulates realistic daily vessel-ping records for each strait with:
- Baseline traffic volumes calibrated to real published figures per strait
- Noise correlated to actual WTI price movements
- Seasonal patterns (spring/autumn peaks)
- Geopolitical shock events (Suez blockage 2021, Hormuz tensions 2019, Houthi attacks 2023, etc.)

**To swap in real AIS data:** call `filter_to_straits(your_df)` then aggregate with `aggregate_to_daily()`.

In [ ]:
# ── Helper: bounding-box filter for real AIS DataFrames ──────────────────
def filter_to_straits(df, lat_col='LAT', lon_col='LON'):
    """Filter a raw AIS DataFrame to rows inside the defined straits.
    Returns filtered frame with a 'strait' column added."""
    masks, names = [], []
    for strait_name, bb in STRAITS.items():
        m = (
            (df[lat_col] >= bb['lat_min']) & (df[lat_col] <= bb['lat_max']) &
            (df[lon_col] >= bb['lon_min']) & (df[lon_col] <= bb['lon_max'])
        )
        masks.append(m)
        names.append(strait_name)
    combined = reduce(lambda a, b: a | b, masks)
    out = df[combined].reset_index(drop=True).copy()
    out['strait'] = np.select(masks, names, default='Unknown')
    return out


def aggregate_to_daily(strait_ais):
    """Aggregate filtered AIS pings to daily per-strait feature columns."""
    strait_ais['date'] = pd.to_datetime(strait_ais['BaseDateTime']).dt.normalize()
    frames = []
    for strait_name in STRAITS:
        s = strait_ais[strait_ais['strait'] == strait_name]
        g = s.groupby('date').agg(
            ship_count         =('MMSI',       'nunique'),
            avg_speed_knots    =('SOG',        'mean'),
            median_speed_knots =('SOG',        'median'),
            std_speed_knots    =('SOG',        'std'),
            avg_draft_m        =('Draft',      'mean'),
            tanker_count       =('VesselType', lambda x: (x == 80).sum()),
            cargo_count        =('VesselType', lambda x: (x == 70).sum()),
            pings              =('MMSI',       'count'),
        ).reset_index()
        g.columns = ['date'] + [f'{strait_name}_{c}' for c in g.columns[1:]]
        frames.append(g)
    daily = reduce(lambda a, b: pd.merge(a, b, on='date', how='outer'), frames)
    return daily.sort_values('date').fillna(0).reset_index(drop=True)


# ── Synthetic generator ───────────────────────────────────────────────────
def make_synthetic_ais(wti_df, date_start, date_end, seed=42):
    rng   = np.random.default_rng(seed)
    dates = pd.date_range(date_start, date_end, freq='D')
    n     = len(dates)

    wti_aligned = (
        wti_df[wti_df['date'].between(date_start, date_end)]
        .set_index('date')
        .reindex(dates).ffill()
        ['oil_price_wti'].values
    )
    wti_norm = (wti_aligned - wti_aligned.mean()) / (wti_aligned.std() + 1e-9)

    doy    = np.array([d.timetuple().tm_yday for d in dates])
    season = 1 + 0.10 * np.sin(2 * np.pi * doy / 365)

    params = {
        'Hormuz':          dict(base=55, trend=0.003, wti_corr= 0.45, tanker_frac=0.55),
        'Bab_el_Mandeb':   dict(base=40, trend=0.002, wti_corr= 0.35, tanker_frac=0.40),
        'Suez_Canal':      dict(base=48, trend=0.002, wti_corr= 0.30, tanker_frac=0.30),
        'Panama_Canal':    dict(base=35, trend=0.001, wti_corr= 0.20, tanker_frac=0.15),
        'Malacca':         dict(base=70, trend=0.004, wti_corr= 0.40, tanker_frac=0.35),
        'Turkish_Straits': dict(base=30, trend=0.001, wti_corr= 0.25, tanker_frac=0.45),
    }

    shocks = [
        ('2019-06-01', '2019-09-30', 'Hormuz',          0.72),
        ('2020-03-01', '2020-06-30', 'Malacca',         0.80),
        ('2020-03-01', '2020-06-30', 'Panama_Canal',    0.78),
        ('2021-03-23', '2021-03-29', 'Suez_Canal',      0.10),
        ('2021-03-30', '2021-05-15', 'Suez_Canal',      0.85),
        ('2022-02-24', '2022-12-31', 'Turkish_Straits', 0.60),
        ('2023-10-01', '2023-12-31', 'Bab_el_Mandeb',   0.50),
    ]

    frames = []
    for strait, p in params.items():
        t     = np.arange(n)
        trend = 1 + p['trend'] * t
        noise = rng.normal(0, 0.08, n)
        ships = np.round(
            p['base'] * trend * season * (1 + p['wti_corr'] * wti_norm + noise)
        ).astype(int).clip(1)

        shock_mult = np.ones(n)
        for s0, s1, ss, mult in shocks:
            if ss == strait:
                shock_mult[(dates >= s0) & (dates <= s1)] = mult
        ships = np.round(ships * shock_mult).astype(int).clip(1)

        tankers = np.round(ships * p['tanker_frac'] * rng.uniform(0.85, 1.15, n)).astype(int).clip(0)
        cargo   = np.round(ships * (1 - p['tanker_frac']) * rng.uniform(0.7, 1.0, n)).astype(int).clip(0)

        frames.append(pd.DataFrame({
            'date':                            dates,
            f'{strait}_ship_count':            ships,
            f'{strait}_tanker_count':          tankers,
            f'{strait}_cargo_count':           cargo,
            f'{strait}_avg_speed_knots':       rng.uniform(7,  15, n).round(2),
            f'{strait}_median_speed_knots':    rng.uniform(6,  14, n).round(2),
            f'{strait}_std_speed_knots':       rng.uniform(0.5, 3, n).round(2),
            f'{strait}_avg_draft_m':           rng.uniform(6,  22, n).round(1),
            f'{strait}_pings':                 ships * rng.integers(4, 12, n),
        }))

    daily = reduce(lambda a, b: pd.merge(a, b, on='date', how='outer'), frames)
    return daily.sort_values('date').reset_index(drop=True)


daily = make_synthetic_ais(wti, DATE_START, DATE_END)
print(f'Synthetic AIS rows : {len(daily):,}')
print(f'Columns ({len(daily.columns)}): {list(daily.columns[:5])} ...')
daily.head()

## 4. Merge Shipping Features with WTI Oil Price

In [ ]:
daily['date'] = pd.to_datetime(daily['date'])
wti['date']   = pd.to_datetime(wti['date'])

merged = pd.merge(daily, wti, on='date', how='inner')
merged = merged.sort_values('date').reset_index(drop=True)

# Column order: shipping vars ... | oil_price_wti | date
feat_cols = [c for c in merged.columns if c not in ('date', 'oil_price_wti')]
merged    = merged[feat_cols + ['oil_price_wti', 'date']]

print(f'Merged shape : {merged.shape}')
print(f'Date range   : {merged["date"].min().date()} to {merged["date"].max().date()}')
merged.head()

## 5. Feature Engineering

In [ ]:
df = merged.copy()

ship_cols   = [c for c in df.columns if 'ship_count'   in c]
tanker_cols = [c for c in df.columns if 'tanker_count' in c]
speed_cols  = [c for c in df.columns if 'avg_speed'    in c]

df['total_ships_global']   = df[ship_cols].sum(axis=1)
df['total_tankers_global'] = df[tanker_cols].sum(axis=1)
df['tanker_ship_ratio']    = (
    df['total_tankers_global'] / df['total_ships_global'].replace(0, np.nan)
).fillna(0)
df['avg_speed_global']     = df[speed_cols].mean(axis=1)

total_ships   = df['total_ships_global']
total_tankers = df['total_tankers_global']

for lag in [1, 3, 7, 14, 30]:
    df[f'ships_lag{lag}']   = total_ships.shift(lag)
    df[f'tankers_lag{lag}'] = total_tankers.shift(lag)

for window in [7, 14, 30, 60]:
    df[f'ships_roll{window}']   = total_ships.rolling(window, min_periods=1).mean()
    df[f'tankers_roll{window}'] = total_tankers.rolling(window, min_periods=1).mean()

for window in [7, 30]:
    df[f'ships_std{window}'] = total_ships.rolling(window, min_periods=1).std().fillna(0)

for strait in STRAITS:
    sc, tc = f'{strait}_ship_count', f'{strait}_tanker_count'
    if sc in df.columns and tc in df.columns:
        df[f'{strait}_tanker_ratio'] = (
            df[tc] / df[sc].replace(0, np.nan)
        ).fillna(0)

df['day_of_week'] = df['date'].dt.dayofweek
df['month']       = df['date'].dt.month
df['quarter']     = df['date'].dt.quarter
df['year']        = df['date'].dt.year
df['day_of_year'] = df['date'].dt.dayofyear

df = df.dropna().reset_index(drop=True)
print(f'Shape after feature engineering: {df.shape}')
print(f'Features: {len([c for c in df.columns if c not in ("date","oil_price_wti")])}')

## 6. Train / Test Split  (80 % train · 20 % test — chronological)

In [ ]:
from sklearn.preprocessing import StandardScaler

TARGET   = 'oil_price_wti'
FEATURES = [c for c in df.columns if c not in (TARGET, 'date')]

X      = df[FEATURES].values
y      = df[TARGET].values
dates_ = df['date'].values

split  = int(len(df) * 0.80)

X_train, X_test = X[:split],  X[split:]
y_train, y_test = y[:split],  y[split:]
dates_train     = dates_[:split]
dates_test      = dates_[split:]

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train : {X_train.shape}  {pd.Timestamp(dates_train[0]).date()} to {pd.Timestamp(dates_train[-1]).date()}')
print(f'Test  : {X_test.shape}   {pd.Timestamp(dates_test[0]).date()} to {pd.Timestamp(dates_test[-1]).date()}')

## 7. Train Models

In [ ]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    'XGBoost': XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        early_stopping_rounds=30, eval_metric='rmse',
        random_state=42, verbosity=0
    ),
    'RandomForest': RandomForestRegressor(
        n_estimators=300, max_depth=10, random_state=42, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42
    ),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...', end=' ')
    if name == 'XGBoost':
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    else:
        model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    results[name] = {'model': model, 'preds': preds, 'mae': mae, 'rmse': rmse, 'r2': r2}
    print(f'MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}')

best_name  = max(results, key=lambda k: results[k]['r2'])
best_preds = results[best_name]['preds']
print(f'\nBest model: {best_name}  (R2 = {results[best_name]["r2"]:.4f})')

## 8. Visualise Predictions vs Actual

In [ ]:
dates_test_dt = pd.to_datetime(dates_test)

fig, axes = plt.subplots(len(models), 1, figsize=(14, 4 * len(models)))
if len(models) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    ax.plot(dates_test_dt, y_test,       label='Actual WTI',   color='#2c7bb6', lw=1.5)
    ax.plot(dates_test_dt, res['preds'], label=f'{name} Pred', color='#d7191c', lw=1.2, alpha=0.85)
    ax.set_title(f"{name}  |  MAE: {res['mae']:.2f}  RMSE: {res['rmse']:.2f}  R2: {res['r2']:.4f}")
    ax.set_ylabel('WTI (USD/bbl)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.3)

plt.suptitle('WTI Oil Price Predictions — Test Set\n(Strategic Straits Shipping Features)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('predictions_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to predictions_plot.png')

## 9. Feature Importance (Best Model)

In [ ]:
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    imp   = pd.Series(best_model.feature_importances_, index=FEATURES)
    top20 = imp.nlargest(20).sort_values()

    fig, ax = plt.subplots(figsize=(10, 6))
    top20.plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'Top 20 Feature Importances — {best_name}')
    ax.set_xlabel('Importance')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to feature_importance.png')

## 10. Build Predictions DataFrame & Export to CSV

In [ ]:
# Unscale test features back to original units
test_feats = pd.DataFrame(
    scaler.inverse_transform(X_test),
    columns=FEATURES
)

pred_cols = [
    pd.Series(res['preds'], name=f'predicted_wti_{name}')
    for name, res in results.items()
]

predictions_df = pd.concat(
    [test_feats]
    + pred_cols
    + [
        pd.Series(y_test,        name='actual_oil_price_wti'),
        pd.Series(dates_test_dt, name='date'),
    ],
    axis=1
)

predictions_df['residual_best_model'] = (
    predictions_df['actual_oil_price_wti']
    - predictions_df[f'predicted_wti_{best_name}']
)

OUT_CSV = 'strait_shipping_oil_predictions.csv'
predictions_df.to_csv(OUT_CSV, index=False)

print(f'Saved to {OUT_CSV}')
print(f'Shape   : {predictions_df.shape}')
print('Columns : shipping features | predicted_wti_* | actual_oil_price_wti | date | residual')
predictions_df.head()

## 11. Model Performance Summary

In [ ]:
summary = pd.DataFrame({
    name: {'MAE ($/bbl)': res['mae'], 'RMSE ($/bbl)': res['rmse'], 'R2': res['r2']}
    for name, res in results.items()
}).T.round(4)

print('=' * 52)
print('  MODEL PERFORMANCE SUMMARY — TEST SET')
print('=' * 52)
print(summary.to_string())
print(f'\nBest model  : {best_name}')
print(f'Test period : {pd.Timestamp(dates_test[0]).date()} to {pd.Timestamp(dates_test[-1]).date()}')
print(f'Output CSV  : {OUT_CSV}  ({os.path.getsize(OUT_CSV)/1024:.1f} KB)')